# AIFM Infrastructure Fund

This notebook presents an infrastructure risk-monitoring workflow for a simulated closed-ended AIF. The fund invests in long-duration real assets across utilities, energy transition, transport, and social infrastructure, held through regulated concessions and contracted structures.

The analysis focuses on asset-level indicators: appraised NAV and its drivers, DSCR and LTV covenant status, concentration, inflation linkage, concession duration, closed-ended funding liquidity, valuation-input stress, and sustainability indicators. Infrastructure data lives in dedicated database tables — not in the shared daily position snapshot.

> **Output gallery:** All tables and plots generated by this notebook are saved in the [fig/AIFM_Infra_Core](../../fig/AIFM_Infra_Core) folder. Readers who prefer to review the generated outputs directly can browse that folder without running the notebook.

In [ ]:
import warnings

from fund_risk_workflow.data.setup_db import run as setup_db

import fund_risk_workflow.data.database as db
import fund_risk_workflow.risk.esg_utils as esg_u
import fund_risk_workflow.ui.print_html_utils as phtml
import fund_risk_workflow.ui.infrastructure_display as ind

warnings.filterwarnings("ignore")

setup_db()
ENGINE = db.get_engine()

## 1. Fund Setup and Risk Policy

### 1.1 Fund Example

The fund profile below sets the operating context for the risk workflow. It defines the strategy, fund type, base currency, reporting setup, and monitoring framework used by the calculations that follow.

In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
FUND_ID = 'AIFM_Infra_Core'
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="01",
)

> Note: Fund characteristics, risk limits, methodologies, and reporting parameters are simulated. They are used to show how a fund-level risk framework can be represented in a structured workflow.

---

### 1.2 Risk Management Policy Parameters

The fund's risk parameters are sourced from the Risk Management Policy configuration. The valuation-input stress scenarios, the CPI + 400bps performance benchmark, the 50bps discount-rate flag, and the covenant watch threshold are documented in the risk policy rather than in notebook code.

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="02",
)

### 1.3 Implementation Context

The analysis is performed as of a fixed valuation date; appraisals, concentration, and ESG indicators use the matching reporting quarter.

In [ ]:
# Fixed valuation date and reporting quarter for all computations
from fund_risk_workflow.config import QUARTER, VALUATION_DATE
VALUATION_DATE, QUARTER

The workflow builder reads the populated infrastructure tables (`infra_funds`, `infra_assets`, `infra_fund_investments`, `infra_cash_flows`, `infra_nav_history`, `infra_valuation_report`, `infra_debt`, `infra_covenants`) and computes every result used in this notebook. From this point onward, code cells contain only display calls.

In [ ]:
# Build the full infrastructure monitoring result set
from fund_risk_workflow.pipeline.infrastructure_workflow import build_infrastructure_workflow

workflow = build_infrastructure_workflow(
    engine=ENGINE,
    fund_id=FUND_ID,
    valuation_date=VALUATION_DATE,
    quarter=QUARTER,
)

---

## 2. Portfolio Overview

An infrastructure fund does not hold daily-priced positions. Data is organised around the asset portfolio: asset master data from the fund administrator, quarterly valuations from independent appraisers, and cash flows between the fund and its investors.

In [ ]:
ind.display_fund_metadata(workflow["portfolio_overview"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="03")

In [ ]:
ind.display_asset_portfolio(workflow["portfolio_overview"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="04")

---

## 3. Valuation and NAV

Assets are valued quarterly by independent appraisers using yield capitalisation (EV = EBITDA / discount rate). For regulated assets the discount rate reflects the allowed WACC; for contracted assets it reflects counterparty credit quality and remaining contract life.

Risk management monitors the NAV trajectory, the asset contribution to NAV, and quarter-on-quarter movements in appraiser discount-rate assumptions. Under AIFMD Article 19, risk management does not challenge appraiser assumptions — it documents movements and escalates changes beyond the policy threshold.

In [ ]:
ind.plot_nav_timeseries(workflow["valuation_summary"]["nav_timeseries"], FUND_ID, valuation_date=VALUATION_DATE, export_id="05")

In [ ]:
ind.plot_nav_by_asset(workflow["valuation_summary"]["asset_breakdown"], FUND_ID, valuation_date=VALUATION_DATE, export_id="06")

In [ ]:
ind.display_discount_rate_movement(workflow["valuation_summary"]["discount_rate_movement"], fund_id=FUND_ID, export_id="07")

---

## 4. Performance Metrics

Infrastructure return metrics mirror the PE convention: MOIC, DPI, and RVPI from cash flows and current NAV, with IRR from XIRR on actual call and distribution dates. The benchmark for a core infrastructure strategy is CPI + 400bps net of fees, per the fund's risk policy.

For a core fund still inside its investment period, high RVPI and low DPI is expected — assets are long-duration and capital is not returned early.

In [ ]:
ind.display_performance(workflow["performance"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="08")

In [ ]:
ind.plot_moic_decomposition(workflow["performance"], FUND_ID, valuation_date=VALUATION_DATE, export_id="09")

---

## 5. Covenant Monitoring

Project-level debt carries DSCR and LTV covenants observed quarterly. The monitor shows the latest actuals against covenant, headroom, breach counts over the trailing window, trend, waiver flags, and a per-asset history sparkline. Headroom below the policy watch threshold renders as Watch.

In [ ]:
ind.display_covenant_monitor(workflow["covenant_monitor"]["dscr"], "DSCR", valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="10")

In [ ]:
ind.display_covenant_monitor(workflow["covenant_monitor"]["ltv"], "LTV", valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="11")

---

## 6. Concentration

Concentration is monitored along sector, country, and sub-type (regulated / contracted / concession-with-volume-risk / merchant). AIFMD does not prescribe sector concentration limits for closed-ended funds; the 40% NAV sector threshold is the fund's own internal policy.

In [ ]:
ind.display_sector_concentration(workflow["concentration"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="12")

In [ ]:
ind.plot_concentration(workflow["concentration"], FUND_ID, valuation_date=VALUATION_DATE, export_id="13")

---

## 7. Inflation and Duration

Two structural characteristics define infrastructure: **inflation linkage** (the share of revenues that adjusts automatically with CPI/PPI) and **long duration** (remaining concession or contract life). Weighted linkage is the NAV-weighted average of asset linkage coefficients. Assets with less than 3 years of remaining concession life are flagged for exit or re-tendering review.

In [ ]:
ind.display_inflation_summary(workflow["inflation_sensitivity"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="14")

In [ ]:
ind.plot_inflation_linkage(workflow["inflation_sensitivity"], FUND_ID, valuation_date=VALUATION_DATE, export_id="15")

In [ ]:
ind.display_duration_profile(workflow["duration_profile"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="16")

In [ ]:
ind.plot_duration_profile(workflow["duration_profile"], FUND_ID, valuation_date=VALUATION_DATE, export_id="17")

---

## 8. Cash Flow and Liquidity

In a closed-ended infrastructure fund, liquidity risk lives on the liability side: capital call obligations, management fees, and asset-level debt service. This is funding liquidity, not asset liquidity — there is no redemption monitoring.

A core fund investing in operational assets generates cash from day one, so the J-curve is shallow. Cashflow coverage above 1.0x means quarterly distributions cover management fees without additional LP capital calls.

In [ ]:
ind.plot_infra_j_curve(workflow["cashflow_profile"], FUND_ID, valuation_date=VALUATION_DATE, export_id="18")

In [ ]:
ind.plot_cashflow_coverage(workflow["cashflow_coverage"], FUND_ID, valuation_date=VALUATION_DATE, export_id="19")

---

## 9. Stress Testing

Infrastructure NAV is driven by the appraiser's discount rate and EBITDA (itself partly a function of inflation linkage). Stress testing here targets valuation model inputs rather than market prices:

$$EV_{stressed} = \frac{EBITDA \times (1 + \lambda \cdot \Delta\pi)}{r + \Delta r}$$

The scenario set (rate shock, inflation loss, combined) is documented in the risk policy (migrated from earlier notebook assumptions, unchanged).

In [ ]:
ind.display_stress_summary(workflow["stress_results"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="20")

In [ ]:
ind.plot_stress_impact(workflow["stress_results"], FUND_ID, valuation_date=VALUATION_DATE, export_id="21")

In [ ]:
ind.display_asset_stress_detail(workflow["stress_results"], "(c) Combined", valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="22")

In [ ]:
ind.plot_asset_stress_detail(workflow["stress_results"], "(c) Combined", FUND_ID, valuation_date=VALUATION_DATE, export_id="23")

---

## 10. Sustainability Risk Indicators

Infrastructure assets are assessed quarterly by independent appraisers alongside the financial valuation; `esg_reporter` identifies the source. ESG profiles vary by sector — renewables and social infrastructure score well while airports and ports carry high carbon intensity. Controversy flags correspond to the covenant events identified in Section 5.

> Scale note: ESG scores use a 0-100 scale, where 100 is best. ESG scores are sustainability-risk inputs and are not mapped to SFDR classifications here.

In [ ]:
esg_u.display_esg_assets(workflow["esg_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="24")

In [ ]:
esg_u.display_esg_summary(workflow["esg_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="25")

In [ ]:
esg_u.plot_esg_profile(workflow["esg_df"], FUND_ID, plot_title='ESG profile — Infrastructure', valuation_date=VALUATION_DATE, export_id="26")

---

## 11. Annex IV Report

Closed-ended infrastructure AIFs report under AIFMD Annex IV with specific treatment:

- **Fund structure**: closed-ended with no periodic redemption; fund life is matched to the concession horizon of the underlying assets.
- **Leverage**: no financial leverage at fund level. Project-level debt is ring-fenced within each SPV and excluded from AIFMD leverage per the project-finance treatment, but disclosed separately.
- **Key metrics**: concession duration, inflation linkage, and covenant status rather than daily VaR. NAV is set by independent appraisers (AIFMD Article 19).

**Regulatory basis:** Delegated Regulation (EU) 231/2013 Article 110 and Annex IV reporting template.

In [ ]:
import fund_risk_workflow.reporting.annex_iv_workflow as annex_iv_workflow
from fund_risk_workflow.pipeline.infrastructure_workflow import INFRA_SECTIONS

annex_iv_result = annex_iv_workflow.run(
    engine=ENGINE,
    fund_id=FUND_ID,
    quarter=QUARTER,
    first_export_id="27",
    sections=INFRA_SECTIONS,
)